# Introduction: building and printing PDE problems

The Poisson problem from section 8 of the plan, written with `qewton.equations`. Everything is imported as `eq`.

In [1]:
import qewton.equations as eq

## Coordinates and sets

A coordinate is a `Variable`. The domain Omega lives in `x`, and its boundary is split into two parts.

In [2]:
x = eq.Variable('x', dim=2)
Omega = eq.Domain('Omega', ambient=x)
dO = eq.Boundary(Omega)
GD = eq.Part('Gamma_D', dO)
GN = eq.Part('Gamma_N', dO)

## Functions

Every function states its inputs. `None` declares a constant.

In [3]:
u = eq.FunctionSymbol(eq.Variable('u'), x)        # unknown, depends on x
f = eq.FunctionSymbol(eq.Variable('f'), None)     # constant source
g = eq.FunctionSymbol(eq.Variable('g'), x)        # Dirichlet data on Gamma_D
h = eq.FunctionSymbol(eq.Variable('h'), x)        # Neumann data on Gamma_N
nu = eq.FunctionSymbol(eq.Variable('nu'), None)   # coefficient, constant in x

## Expressions

Operators come from `eq`. `print` shows them as math, and a bare expression shows the same text.

In [4]:
flux = nu * eq.grad(u, x)
lhs = -eq.div(flux, x)
print(lhs)

−∇_x·(nu·∇_x u)


In [6]:
eq.dot(eq.Restrict(flux, GN), eq.Normal(GN))

dot((nu·∇_x u)|_{Gamma_N}, n_{Gamma_N})

## Constraints and the problem

`Eq(lhs, rhs).on(set)` attaches the set where the constraint holds. `Problem` refuses a constraint that has no set.

In [7]:
problem = eq.Problem(
    constraints=(
        eq.Eq(lhs, f).on(Omega),
        eq.Eq(eq.Restrict(u, GD), g).on(GD),
        eq.Eq(eq.dot(eq.Restrict(flux, GN), eq.Normal(dO)), h).on(GN),
    ),
    partitions=(eq.Partition(dO, (GD, GN)),),
)
print(problem)

constraints:
  1. −∇_x·(nu·∇_x u) = f on Omega
  2. u|_{Gamma_D} = g on Gamma_D
  3. dot((nu·∇_x u)|_{Gamma_N}, n_{∂Omega}) = h on Gamma_N
partitions:
  ∂Omega = Gamma_D ⊔ Gamma_N


In [8]:
print('functions:', sorted(s.name for s in problem.symbols()))
print('sets:', sorted(str(s) for s in problem.sets()))

functions: ['f', 'g', 'h', 'nu', 'u']
sets: ['Gamma_D', 'Gamma_N', 'Omega', '∂Omega']


## Time-dependent problems

The heat equation has a time variable. Declaring it with `time=` tells the analysis which variable evolves. The initial condition lives on the slice `t = 0`.

In [9]:
t = eq.Variable('t', dim=1)
Interval = eq.Domain('I', ambient=t)
w = eq.FunctionSymbol(eq.Variable('w'), x * t)   # depends on x and t, in this order
w0 = eq.FunctionSymbol(eq.Variable('w0'), x)     # initial data
space_time = eq.Product((Omega, Interval))
initial = eq.Product((Omega, eq.Slice(t, 0.0)))
heat = eq.Problem(
    constraints=(
        eq.Eq(eq.dt(w, t) - eq.lap(w, x), f).on(space_time),
        eq.Eq(eq.Restrict(w, initial), w0).on(initial),
    ),
    time=t,
)
heat

constraints:
  1. ∂w/∂t − Δ_x w = f on Omega × I
  2. w|_{Omega × {t = 0}} = w0 on Omega × {t = 0}
time: t

## Next steps

The analysis passes (validation, shapes, classification) and the report come in the next milestones. This notebook only builds and prints problems.